# PROSPER Demo Notebook

## Pediatric Pathology Slide Search Tool

This notebook demonstrates the end-to-end PROSPER pipeline using:
- **Virchow2** patch embedding model
- **SAMPLER (Percentile)** slide embedding method

### What you'll learn:
1. **Setup & Verification** - Verify your environment is ready
2. **End-to-End Pipeline** - Process a WSI from scratch
3. **Slide Search** - Find morphologically similar slides with visualization
4. **ROI Search** - Find similar tissue regions with heatmap overlays
5. **OT Slide Comparison** - Optimal Transport patch-level similarity
6. **Bayesian Disease Probabilities** - Estimate disease likelihoods from neighbors
7. **LLM Clinical Summarization** - Generate grounded, clinician-friendly reports

---

## 1. Setup & Verification

First, let's verify that your environment is correctly set up.

In [ ]:
# Environment check
import sys
import os

print("PROSPER Environment Check")
print("=" * 50)
print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {os.getcwd()}")

# Check GPU
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected. CPU mode will be used (slower).")

In [ ]:
# Check required packages
required_packages = [
    ('openslide', 'openslide-python + openslide-bin'),
    ('timm', 'timm'),
    ('huggingface_hub', 'huggingface_hub'),
]

print("Checking required packages...")
for pkg, install_name in required_packages:
    try:
        __import__(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg} - Install with: pip install {install_name}")

# Check HuggingFace authentication (required for Virchow2)
print("\nChecking HuggingFace authentication...")
try:
    from huggingface_hub import HfApi
    api = HfApi()
    user = api.whoami()
    print(f"  [OK] Logged in as: {user['name']}")
except Exception as e:
    print(f"  [WARNING] Not authenticated. Run: hf auth login")
    print(f"  Virchow2 is a gated model - you need to request access and authenticate.")

---
## 2. Configuration

Set up paths for the demo. We'll use the sample data in the `data/` folder.

In [ ]:
import os
import sys
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial.distance import cdist

# Compatibility shim: reference pickle saved with numpy ≥ 2.x
try:
    import numpy._core.numeric          # noqa: F401
except ImportError:
    import numpy.core.numeric as _numeric
    sys.modules["numpy._core.numeric"] = _numeric

# Configuration
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"

# Input files
SLIDE_PATH = DATA_DIR / "4261.svs"  # Sample WSI
REFERENCE_MATRIX_PATH = DATA_DIR / "final_slides_sampler_matrix.pkl"  # Slide embeddings database
METADATA_PATH = DATA_DIR / "final_scan_id_disease_map.csv"  # Disease labels
RICH_METADATA_PATH = DATA_DIR / "final_scan_id_disease_map_rich_data.csv"  # Rich metadata for LLM
REFERENCE_PATCHES_DIR = DATA_DIR / "patch_embeddings"  # Patch embeddings for ROI search

# Heatmap visualization files
MOSAIC_METADATA_PATH = DATA_DIR / "mosaic_metadata.pkl"  # Mosaic cluster locations
THUMBNAILS_DIR = DATA_DIR / "thumbnails"  # Slide thumbnails for overlay
DIMENSIONS_CSV = DATA_DIR / "slide_dimensions.csv"  # Slide dimensions for coordinate mapping

# Output directories
MOSAIC_DIR = OUTPUT_DIR / "mosaic"
PATCH_EMB_DIR = OUTPUT_DIR / "patch_embeddings"
SLIDE_EMB_DIR = OUTPUT_DIR / "slide_embeddings"
SEARCH_RESULTS_DIR = OUTPUT_DIR / "search_results"
HEATMAP_DIR = OUTPUT_DIR / "heatmaps"
BAYESIAN_DIR = OUTPUT_DIR / "bayesian"
LLM_DIR = OUTPUT_DIR / "llm"

# Create output directories
for d in [MOSAIC_DIR, PATCH_EMB_DIR, SLIDE_EMB_DIR, SEARCH_RESULTS_DIR,
          HEATMAP_DIR, BAYESIAN_DIR, LLM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Slide ID (extracted from filename)
SLIDE_ID = SLIDE_PATH.stem

print(f"Slide ID: {SLIDE_ID}")
print(f"Slide path: {SLIDE_PATH}")
print(f"Slide exists: {SLIDE_PATH.exists()}")
print(f"\nHeatmap resources:")
print(f"  Mosaic metadata: {MOSAIC_METADATA_PATH.exists()}")
print(f"  Thumbnails dir: {THUMBNAILS_DIR.exists()}")
print(f"  Dimensions CSV: {DIMENSIONS_CSV.exists()}")
print(f"\nLLM resources:")
print(f"  Rich metadata: {RICH_METADATA_PATH.exists()}")

---
## 3. End-to-End Pipeline

The pipeline consists of 8 steps:
1. **Mosaic Generation** - Extract tissue patches from WSI
2. **Patch Embedding** - Extract Virchow2 embeddings for each patch
3. **Slide Embedding** - Aggregate patches into slide-level embedding (percentile method)
4. **Slide Search** - Find morphologically similar slides
5. **ROI Search** - Find similar tissue regions with heatmap overlays
6. **OT Slide Comparison** - Optimal Transport patch-level similarity
7. **Bayesian Disease Probabilities** - Estimate disease likelihoods from neighbors
8. **LLM Clinical Summarization** - Generate grounded, clinician-friendly case reports

### Step 1: Mosaic Generation

Extract tissue patches from the whole slide image.

In [ ]:
from prosper.preprocessing.mosaic import MosaicGenerator

# Initialize mosaic generator
mosaic_gen = MosaicGenerator(
    base_patchsize=512,
    num_clusters=100,  # Number of representative patches to select
    verbose=True
)

# Generate mosaic
mosaic_output = MOSAIC_DIR / f"{SLIDE_ID}"
mosaic_output.mkdir(exist_ok=True)

print(f"Processing slide: {SLIDE_PATH}")
mosaic = mosaic_gen.process_slide(
    slide_fn=str(SLIDE_PATH),
    output_dir=str(mosaic_output)
)

print(f"\nMosaic generated: {len(mosaic)} patches selected")
print(f"Sample patch location: {mosaic[0]['wsi_loc']}")

### Step 2: Patch Embedding Extraction

Extract Virchow2 embeddings for each patch.

In [ ]:
from prosper.embeddings.patch_embedding import Virchow2EmbeddingExtractor

# Initialize Virchow2 extractor
virchow2 = Virchow2EmbeddingExtractor(
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=64,
    verbose=True
)

# Extract embeddings
print(f"Extracting Virchow2 embeddings for {len(mosaic)} patches...")
patch_embeddings = virchow2.extract_slide_embeddings(
    slide_path=str(SLIDE_PATH),
    mosaic=mosaic
)

print(f"\nPatch embeddings shape: {patch_embeddings.shape}")
print(f"Embedding dimension: {patch_embeddings.shape[1]}")

# Save patch embeddings (in tmp subdirectory, matching pipeline structure)
patch_emb_subdir = PATCH_EMB_DIR / "tmp"
patch_emb_subdir.mkdir(exist_ok=True)
patch_emb_path = patch_emb_subdir / f"{SLIDE_ID}.pkl"
with open(patch_emb_path, 'wb') as f:
    pickle.dump((patch_embeddings, mosaic), f)
print(f"Saved to: {patch_emb_path}")

### Step 3: Slide Embedding Generation

Aggregate patch embeddings into a slide-level representation using the percentile method.

In [ ]:
from prosper.embeddings.slide_embedding import PercentileSlideEmbeddingGenerator

# Initialize percentile embedding generator
percentile_gen = PercentileSlideEmbeddingGenerator(
    feature_dim=1280,  # Virchow2 class token dimension
    verbose=True
)

# Generate slide embedding
slide_embedding = percentile_gen.process_single_slide(patch_embeddings)

print(f"Slide embedding shape: {slide_embedding.shape}")
print(f"  - 10 percentiles x 1280 features = {10 * 1280} dimensions (flattened)")

# Save slide embedding
slide_emb_path = SLIDE_EMB_DIR / f"{SLIDE_ID}.pkl"
with open(slide_emb_path, 'wb') as f:
    pickle.dump(slide_embedding, f)
print(f"Saved to: {slide_emb_path}")

---
## 4. Slide Search

Find morphologically similar slides from the reference database.

In [ ]:
# Load reference database
print("Loading reference database...")
with open(REFERENCE_MATRIX_PATH, 'rb') as f:
    ref_df = pickle.load(f)

ref_slide_ids = [str(sid) for sid in ref_df.index.tolist()]
ref_matrix = ref_df.values

print(f"Reference database: {len(ref_slide_ids)} slides")
print(f"Embedding dimension: {ref_matrix.shape[1]}")

# Load metadata for disease labels
metadata = pd.read_csv(METADATA_PATH)
disease_map = dict(zip(metadata['Slide.Scan.File'].astype(str), metadata['Tumor.group']))

# Check query slide's disease
query_disease = disease_map.get(SLIDE_ID, 'Unknown')
print(f"\nQuery slide {SLIDE_ID} disease: {query_disease}")

In [ ]:
# Load slide embedding if not already in memory
try:
    slide_embedding
except NameError:
    slide_emb_path = SLIDE_EMB_DIR / f"{SLIDE_ID}.pkl"
    with open(slide_emb_path, 'rb') as f:
        slide_embedding = pickle.load(f)
    print(f"Loaded slide embedding from: {slide_emb_path}")

# Perform slide search
query = slide_embedding.flatten().reshape(1, -1)

# Compute L1 (cityblock) distances
distances = cdist(query, ref_matrix, metric='cityblock')[0]
sorted_idx = np.argsort(distances)

# Get top-20 results (excluding self)
print("\nTop-20 Similar Slides:")
print("-" * 60)
print(f"{'Rank':<6}{'Slide ID':<12}{'Disease':<15}{'Distance':<12}{'Match'}")
print("-" * 60)

results = []
rank = 0
for idx in sorted_idx:
    sid = ref_slide_ids[idx]
    if sid == SLIDE_ID:
        continue
    rank += 1
    if rank > 20:
        break
    disease = disease_map.get(sid, 'Unknown')
    match = "*" if disease == query_disease else ""
    print(f"{rank:<6}{sid:<12}{disease:<15}{distances[idx]:<12.2f}{match}")
    results.append({
        'query_slide_id': SLIDE_ID,
        'hit_slide_id': sid,
        'distance': distances[idx],
        'rank': rank,
        'hit_disease': disease
    })

# Summary
same_disease_count = sum(1 for r in results if r['hit_disease'] == query_disease)
print(f"\nRetrieval rate ({query_disease}): {same_disease_count}/20 = {same_disease_count/20*100:.0f}%")

In [ ]:
# Save slide search results
slide_search_df = pd.DataFrame(results)
slide_search_path = SEARCH_RESULTS_DIR / f"{SLIDE_ID}_slide_search.csv"
slide_search_df.to_csv(slide_search_path, index=False)
print(f"Saved slide search results to: {slide_search_path}")
slide_search_df.head(10)

### 4.1 Slide Search Visualization

Visualize the slide search results - disease distribution and distance patterns.

In [ ]:
import matplotlib.pyplot as plt

# Disease distribution in top-20 results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Disease distribution in slide search results
ax1 = axes[0]
disease_counts = slide_search_df['hit_disease'].value_counts()
colors = ['#2ecc71' if d == query_disease else '#3498db' for d in disease_counts.index]
ax1.bar(disease_counts.index, disease_counts.values, color=colors)
ax1.set_xlabel('Disease Type')
ax1.set_ylabel('Count')
ax1.set_title(f'Slide Search Results\n(Query: {SLIDE_ID}, Disease: {query_disease})')
ax1.tick_params(axis='x', rotation=45)

# Plot 2: Distance distribution
ax2 = axes[1]
same_disease = slide_search_df[slide_search_df['hit_disease'] == query_disease]['distance']
diff_disease = slide_search_df[slide_search_df['hit_disease'] != query_disease]['distance']

if len(same_disease) > 0:
    ax2.hist(same_disease, bins=10, alpha=0.7, label=f'Same ({query_disease})', color='#2ecc71')
if len(diff_disease) > 0:
    ax2.hist(diff_disease, bins=10, alpha=0.7, label='Different', color='#e74c3c')
ax2.set_xlabel('L1 Distance')
ax2.set_ylabel('Count')
ax2.set_title('Distance Distribution by Disease Match')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 5. ROI Search

Find similar tissue regions (patches) across the reference database.

This is useful for:
- Finding morphologically similar regions in other slides
- Identifying diagnostic patterns
- Quality control and annotation transfer

In [ ]:
from prosper.search import ROISearchEngine

# Initialize ROI search engine
roi_engine = ROISearchEngine(
    top_k=5,  # Top-5 similar patches per query patch
    verbose=True
)

# Load reference patch embeddings
print("Loading reference patch embeddings...")
roi_engine.load_reference_database(
    embeddings_dir=str(REFERENCE_PATCHES_DIR)
)

print(f"Loaded {len(roi_engine.reference_embeddings)} slides for ROI search")

In [ ]:
# Load patch embeddings if not already in memory
# (In case you're running this cell independently)
try:
    patch_embeddings
except NameError:
    patch_emb_path = PATCH_EMB_DIR / "tmp" / f"{SLIDE_ID}.pkl"
    with open(patch_emb_path, 'rb') as f:
        patch_embeddings, mosaic = pickle.load(f)
    print(f"Loaded patch embeddings from: {patch_emb_path}")

# Search for similar patches (using first 10 patches as demo)
n_query_patches = min(10, len(mosaic))  # Limit for demo speed
query_coords = np.array([m['wsi_loc'] for m in mosaic])

print(f"Searching for {n_query_patches} query patches...")
roi_results = roi_engine.search(
    query_embeddings=patch_embeddings,
    query_slide_id=SLIDE_ID,
    query_coords=query_coords,
    patch_indices=list(range(n_query_patches))  # First N patches
)

In [ ]:
# Display ROI search results
print("\nROI Search Results (Top-5 per patch):")
print("-" * 80)

for patch_idx, patch_results in enumerate(roi_results[:5]):  # Show first 5 patches
    query_loc = query_coords[patch_idx]
    print(f"\nQuery Patch {patch_idx} @ ({query_loc[0]}, {query_loc[1]}):")
    for result in patch_results[:3]:  # Top-3 matches
        hit_disease = disease_map.get(result.hit_slide_id, 'Unknown')
        loc_str = f"({result.hit_location[0]}, {result.hit_location[1]})" if result.hit_location else "N/A"
        print(f"  Rank {result.rank}: Slide {result.hit_slide_id} ({hit_disease}) @ {loc_str}, dist={result.distance:.2f}")

In [ ]:
# Save ROI search results
roi_df = roi_engine.results_to_dataframe(roi_results)

# Add disease labels
roi_df['hit_disease'] = roi_df['hit_slide_id'].map(disease_map)

roi_search_path = SEARCH_RESULTS_DIR / f"{SLIDE_ID}_roi_search.csv"
roi_df.to_csv(roi_search_path, index=False)
print(f"Saved ROI search results to: {roi_search_path}")
print(f"Total results: {len(roi_df)} rows")

roi_df.head(10)

### 5.1 ROI Heatmap Visualization

Generate a heatmap showing the similarity of each patch in a matched slide to a query patch.
The heatmap is overlaid on the slide thumbnail for context.

In [ ]:
from prosper.visualization import generate_roi_search_heatmap
from PIL import Image

# Select a query patch and its top hit for heatmap generation
query_patch_idx = 0  # First query patch
top_hit = roi_results[query_patch_idx][0]  # Top-1 match

print(f"Query: Slide {SLIDE_ID}, Patch {query_patch_idx}")
print(f"Hit: Slide {top_hit.hit_slide_id}, Location {top_hit.hit_location}")
print(f"Distance: {top_hit.distance:.2f}")

# Generate heatmap with thumbnail overlay
result_paths = generate_roi_search_heatmap(
    embeddings_dir=str(REFERENCE_PATCHES_DIR),
    query_slide_id=SLIDE_ID,
    query_patch_idx=query_patch_idx,
    hit_slide_id=top_hit.hit_slide_id,
    output_dir=str(HEATMAP_DIR),
    mosaic_path=str(MOSAIC_METADATA_PATH),
    thumbnails_dir=str(THUMBNAILS_DIR),
    dimensions_csv=str(DIMENSIONS_CSV),  # For accurate coordinate mapping
    patch_size=512,
    tile_size=16,
    overlay_alpha=120
)

print(f"\nGenerated heatmap: {result_paths.get('heatmap')}")
print(f"Generated overlay: {result_paths.get('overlay')}")

In [ ]:
# Display the heatmap overlay
if 'overlay' in result_paths:
    overlay_img = Image.open(result_paths['overlay'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Standalone heatmap
    heatmap_img = Image.open(result_paths['heatmap'])
    axes[0].imshow(heatmap_img)
    axes[0].set_title(f'Similarity Heatmap\n(Query: {SLIDE_ID} → Hit: {top_hit.hit_slide_id})')
    axes[0].axis('off')
    
    # Overlay on thumbnail
    axes[1].imshow(overlay_img)
    axes[1].set_title(f'Heatmap Overlay on Thumbnail\n(Warmer colors = More similar)')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nColor scale: Red (most similar) → Violet (least similar)")
else:
    print("Overlay not generated (thumbnail may be missing)")

---
## 6. OT (Optimal Transport) Slide Comparison

Compare two slides at the patch level using Optimal Transport.
This computes a transport plan between patch distributions and gives
per-patch similarity scores, which can then be visualized as heatmaps.

**Requirements:** `pip install POT` (Python Optimal Transport library)

In [ ]:
from prosper.embeddings.slide_embedding import OTSlideEmbeddingGenerator
from prosper.visualization.ot_visualization import (
    compute_per_patch_weighted_distance_scores,
    normalize_scores
)

# Pick the top-1 hit slide from slide search for OT comparison
ot_hit_slide_id = results[0]['hit_slide_id']
print(f"OT comparison: Query slide {SLIDE_ID} vs Hit slide {ot_hit_slide_id}")

# Load patch embeddings for both slides
from prosper.visualization.roi_heatmap import load_embedding_file, load_mosaic_metadata, load_slide_dimensions

query_embs, query_coords = load_embedding_file(str(REFERENCE_PATCHES_DIR), SLIDE_ID)
hit_embs, hit_coords = load_embedding_file(str(REFERENCE_PATCHES_DIR), ot_hit_slide_id)

print(f"\nQuery slide {SLIDE_ID}: {query_embs.shape[0]} patches, embedding dim {query_embs.shape[1]}")
print(f"Hit slide {ot_hit_slide_id}: {hit_embs.shape[0]} patches, embedding dim {hit_embs.shape[1]}")

# Load mosaic metadata for both slides (needed for heatmap visualization)
query_mosaic = load_mosaic_metadata(str(MOSAIC_METADATA_PATH), SLIDE_ID)
hit_mosaic = load_mosaic_metadata(str(MOSAIC_METADATA_PATH), ot_hit_slide_id)
print(f"\nQuery mosaic tiles: {len(query_mosaic) if query_mosaic else 0}")
print(f"Hit mosaic tiles: {len(hit_mosaic) if hit_mosaic else 0}")

# Initialize OT generator
ot_gen = OTSlideEmbeddingGenerator(
    metric="cosine",
    reg=0.0,          # Exact EMD (set > 0 for entropic regularization = faster)
    feature_dim=1280,  # Virchow2 class token dimension
    verbose=True
)

# Process both slides (normalize features)
ot_query_data = ot_gen.process_single_slide(query_embs.astype(np.float32), SLIDE_ID, query_mosaic)
ot_hit_data = ot_gen.process_single_slide(hit_embs.astype(np.float32), ot_hit_slide_id, hit_mosaic)

# Compute OT distance with transport matrix
print("\nComputing Optimal Transport distance (this may take a moment)...")
ot_result = ot_gen.compute_detailed_similarity(ot_query_data, ot_hit_data)

print(f"\n  OT (Wasserstein) distance: {ot_result['distance']:.6f}")
print(f"  Transport matrix T shape: {ot_result['T'].shape}")
print(f"  Cost matrix M shape: {ot_result['M'].shape}")

# Compute per-patch weighted distance scores
T = ot_result['T']
M = ot_result['M']
D_query, D_hit = compute_per_patch_weighted_distance_scores(T, M)

print(f"\nPer-patch scores (lower = more similar):")
print(f"  Query slide: min={D_query.min():.6f}, max={D_query.max():.6f}, mean={D_query.mean():.6f}")
print(f"  Hit slide:   min={D_hit.min():.6f}, max={D_hit.max():.6f}, mean={D_hit.mean():.6f}")

### 6.1 OT Heatmap Visualization

Generate per-patch OT similarity heatmaps for both slides, overlaid on thumbnails.
Warmer colors indicate patches that contributed more to the overall slide similarity.

In [ ]:
from prosper.visualization.roi_heatmap import (
    load_thumbnail, get_slide_dimensions,
    generate_heatmap_for_thumbnail, overlay_heatmap_on_thumbnail as roi_overlay
)

OT_HEATMAP_DIR = OUTPUT_DIR / "ot_heatmaps"
OT_HEATMAP_DIR.mkdir(parents=True, exist_ok=True)

# Load thumbnails and slide dimensions for both slides
thumb_query = load_thumbnail(str(THUMBNAILS_DIR), SLIDE_ID)
thumb_hit = load_thumbnail(str(THUMBNAILS_DIR), ot_hit_slide_id)
dims_query = get_slide_dimensions(SLIDE_ID, str(DIMENSIONS_CSV))
dims_hit = get_slide_dimensions(ot_hit_slide_id, str(DIMENSIONS_CSV))

print(f"Query thumbnail: {thumb_query.size if thumb_query else 'Not found'}")
print(f"Hit thumbnail: {thumb_hit.size if thumb_hit else 'Not found'}")
print(f"Query slide dimensions: {dims_query}")
print(f"Hit slide dimensions: {dims_hit}")

# Generate aligned heatmaps in thumbnail space and overlay
ot_overlays = {}
for label, mosaic_tiles, scores, thumbnail, slide_dims, sid, other_sid in [
    ("query", query_mosaic, D_query, thumb_query, dims_query, SLIDE_ID, ot_hit_slide_id),
    ("hit", hit_mosaic, D_hit, thumb_hit, dims_hit, ot_hit_slide_id, SLIDE_ID),
]:
    if mosaic_tiles is None or thumbnail is None:
        print(f"  Skipping {label}: missing mosaic or thumbnail")
        continue

    # Generate standalone heatmap (for reference)
    from prosper.visualization.roi_heatmap import generate_roi_heatmap
    heatmap_path = str(OT_HEATMAP_DIR / f"ot_heatmap_{sid}_vs_{other_sid}.png")
    heatmap_img = generate_roi_heatmap(
        mosaic_tiles=mosaic_tiles,
        distances=scores,  # OT weighted distance scores
        patch_size=512,
        tile_size=16,
        output_path=heatmap_path
    )

    # Generate overlay on thumbnail
    overlay_path = str(OT_HEATMAP_DIR / f"ot_overlay_{sid}_vs_{other_sid}.png")
    overlay_img = roi_overlay(
        heatmap_image=heatmap_img,
        thumbnail=thumbnail,
        alpha=120,
        output_path=overlay_path,
        mosaic_tiles=mosaic_tiles,
        distances=scores,
        patch_size=512,
        slide_dimensions=slide_dims
    )
    ot_overlays[label] = {'heatmap': heatmap_img, 'overlay': overlay_img, 'sid': sid}

print(f"\nGenerated OT heatmaps in: {OT_HEATMAP_DIR}")

In [ ]:
# Display OT heatmap overlays side-by-side
if len(ot_overlays) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, (label, data) in zip(axes, ot_overlays.items()):
        ax.imshow(data['overlay'])
        disease = disease_map.get(data['sid'], 'Unknown')
        ax.set_title(f"Slide {data['sid']} ({disease})\nOT Weighted Distance Heatmap")
        ax.axis('off')

    fig.suptitle(
        f"OT Slide Comparison: {SLIDE_ID} vs {ot_hit_slide_id}  |  "
        f"Wasserstein Distance = {ot_result['distance']:.4f}",
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

    print("Color scale: Red (most similar / lowest OT cost) → Violet (least similar)")
else:
    print("Could not generate both OT overlays (missing thumbnails or mosaic data)")

---
## 7. Bayesian Disease Probabilities

Estimate disease likelihoods for the query slide using exponential-weighted
nearest neighbors from the full distance matrix.

The Bayesian module converts raw distances into a probability distribution
over all disease types, controlled by two parameters:
- **n_neighbors**: how many neighbors to consider (default: 20)
- **temperature**: smoothing parameter — larger values produce softer distributions (default: 20)

In [ ]:
from prosper.bayesian import compute_disease_probs

# Build full pairwise distance matrix from the reference embeddings
# (query slide is already part of the reference set for this demo)
print("Computing Bayesian disease probabilities...")
print("=" * 60)

# Full L1 distance matrix over all reference slides
full_distances = cdist(ref_matrix, ref_matrix, metric='cityblock')

# Class labels aligned with ref_slide_ids
class_labels = [disease_map.get(sid, 'UNKNOWN') for sid in ref_slide_ids]

# Parameters
N_NEIGHBORS = 20
TEMPERATURE = 20.0

list_of_dicts, prob_df = compute_disease_probs(
    distance_matrix=full_distances,
    class_labels=class_labels,
    n_neighbors=N_NEIGHBORS,
    T=TEMPERATURE,
    test_slide_ids=ref_slide_ids,
)

# Extract probabilities for the query slide
query_idx = ref_slide_ids.index(SLIDE_ID)
query_probs = list_of_dicts[query_idx]

# Keep only non-zero probabilities, sorted descending
nonzero_probs = {d: p for d, p in sorted(query_probs.items(), key=lambda x: -x[1]) if p > 1e-8}

print(f"\nDisease probabilities for slide {SLIDE_ID} (top entries):")
print("-" * 40)
for disease, prob in list(nonzero_probs.items())[:10]:
    marker = " ← query disease" if disease == query_disease else ""
    print(f"  {disease:<20s} {prob:.6f}{marker}")

# Save Bayesian results
bayesian_json_path = BAYESIAN_DIR / f"disease_probs_{SLIDE_ID}.json"
with open(bayesian_json_path, 'w') as f:
    json.dump({
        'slide_id': SLIDE_ID,
        'n_neighbors': N_NEIGHBORS,
        'temperature': TEMPERATURE,
        'probabilities': nonzero_probs,
    }, f, indent=2)
print(f"\nSaved Bayesian probabilities to: {bayesian_json_path}")

bayesian_csv_path = BAYESIAN_DIR / f"disease_probs_{SLIDE_ID}.csv"
prob_df.loc[[SLIDE_ID]].to_csv(bayesian_csv_path)
print(f"Saved probability CSV to: {bayesian_csv_path}")

In [ ]:
# Visualize Bayesian disease probability distribution
fig, ax = plt.subplots(figsize=(10, 5))

diseases_sorted = list(nonzero_probs.keys())
probs_sorted = list(nonzero_probs.values())
colors = ['#2ecc71' if d == query_disease else '#3498db' for d in diseases_sorted]

ax.barh(diseases_sorted[::-1], probs_sorted[::-1], color=colors[::-1])
ax.set_xlabel('Posterior Probability')
ax.set_title(f'Bayesian Disease Probabilities for Slide {SLIDE_ID}\n'
             f'(n_neighbors={N_NEIGHBORS}, temperature={TEMPERATURE})')
ax.set_xlim(0, 1.05)

# Add probability values as text
for i, (d, p) in enumerate(zip(diseases_sorted[::-1], probs_sorted[::-1])):
    ax.text(p + 0.01, i, f'{p:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Green = query disease ({query_disease}), Blue = other diseases")

---
## 8. LLM Clinical Summarization

Generate a grounded, clinician-friendly narrative from the search results using an LLM.

The summarization pipeline has two steps:
1. **Build LLM-ready JSON** — Combine neighbor metadata, similarity scores, and Bayesian
   probabilities into a structured payload (`prosper.llm.build_json_for_slide`)
2. **Generate summary** — Send the payload to a GPT model with a grounded prompt that
   enforces inline citations, factual claims verification, and provenance checks
   (`prosper.llm.attempt_llm_summary_from_payload`)

**Requirements:**
- Azure OpenAI or OpenAI API key set in environment variables
- Rich metadata CSV with clinical columns (age, sex, site, treatment, etc.)

### 8.1 Build LLM-Ready JSON Payload

Construct a structured JSON with the query slide's metadata, its top-k neighbors,
and the Bayesian disease probabilities.

In [ ]:
from prosper.llm import load_results_new, load_metadata, build_json_for_slide

print("Building LLM-ready JSON payload...")
print("=" * 60)

# Load similarity matrix and rich metadata
sim_matrix = load_results_new(str(REFERENCE_MATRIX_PATH))
rich_metadata = load_metadata(str(RICH_METADATA_PATH))

# Build the structured JSON payload
K_NEIGHBORS = 5
payload = build_json_for_slide(sim_matrix, rich_metadata, slide_id=SLIDE_ID, k=K_NEIGHBORS)

# Attach Bayesian probabilities
payload['probabilities'] = nonzero_probs

# Save the payload
llm_json_path = LLM_DIR / f"slide_{SLIDE_ID}.json"
with open(llm_json_path, 'w') as f:
    json.dump(payload, f, indent=2)

print(f"\nPayload saved to: {llm_json_path}")
print(f"\nCurrent case: {payload['current_case']['case_id']}")
print(f"  Diagnosis: {payload['current_case'].get('diagnosis', 'N/A')}")
print(f"  Tumor group: {payload['current_case'].get('tumor_group', 'N/A')}")
print(f"  Treatment: {payload['current_case'].get('treatment', 'N/A')}")
print(f"  Discrepant: {payload['current_case'].get('discrepant', 'N/A')}")
print(f"\nNeighbors ({K_NEIGHBORS}):")
for i, n in enumerate(payload['neighbors'], 1):
    print(f"  {i}. {n['case_id']} ({n.get('tumor_group', '?')}) - distance: {n['similarity']:.2f}")
print(f"\nBayesian probabilities (top 3):")
for d, p in list(nonzero_probs.items())[:3]:
    print(f"  {d}: {p:.4f}")

### 8.2 Generate LLM Summary

Send the payload to the LLM for grounded summarization. The model produces:
- A clinician-friendly narrative (≤200 words) with inline `[case_id]` citations
- Structured claims for post-hoc verification
- Treatment status, discrepant flag, and Bayesian probabilities in the opening sentence

> **Note:** Requires `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_API_VERSION`,
> and `AZURE_OPENAI_DEPLOYMENT` environment variables to be set (or `OPENAI_API_KEY` for OpenAI).

In [ ]:
from prosper.llm import attempt_llm_summary_from_payload, MEDICAL_DISCLAIMER

# Load the payload we just saved (or use the in-memory one)
with open(llm_json_path, 'r') as f:
    llm_payload = json.load(f)

print("Generating LLM clinical summary...")
print("=" * 60)

llm_text, accepted = attempt_llm_summary_from_payload(llm_payload)

if llm_text and not llm_text.startswith("ERROR:"):
    label = "LLM Summary" if accepted else "LLM Summary (truncated)"
    print(f"\n{label}:\n")
    print(llm_text)
    print(f"\n{MEDICAL_DISCLAIMER}")
    
    # Save the summary
    summary_path = LLM_DIR / f"summary_{SLIDE_ID}.txt"
    with open(summary_path, 'w') as f:
        f.write(f"{label}: {llm_text}\n\n{MEDICAL_DISCLAIMER}\n")
    print(f"\nSaved summary to: {summary_path}")
elif llm_text:
    print(f"\n{llm_text}")
    print("\nFalling back to deterministic summary...")
else:
    print("\nLLM not available (no API key set). Set AZURE_OPENAI_* or OPENAI_API_KEY env vars.")
    print("You can still use the JSON payload for manual review or other tools.")

---
## Summary

This notebook demonstrated the complete PROSPER pipeline:

1. **Mosaic Generation** - Extracted tissue patches from WSI
2. **Patch Embedding** - Generated Virchow2 embeddings (first 1280 features used)
3. **Slide Embedding** - Aggregated to slide-level using SAMPLER/percentiles (10 x 1280 = 12800-dim)
4. **Slide Search** - Found morphologically similar slides with visualization
5. **ROI Search** - Found similar tissue regions with heatmap overlays
6. **OT Slide Comparison** - Computed Optimal Transport between slides with per-patch heatmaps
7. **Bayesian Disease Probabilities** - Estimated disease likelihoods from distance-weighted neighbors
8. **LLM Clinical Summarization** - Generated grounded, clinician-friendly case reports with citations

### Output Files
- `output/mosaic/{slide_id}/` - Mosaic patches and metadata
- `output/patch_embeddings/{slide_id}.pkl` - Patch embeddings
- `output/slide_embeddings/{slide_id}.pkl` - Slide embedding
- `output/search_results/{slide_id}_slide_search.csv` - Slide search results
- `output/search_results/{slide_id}_roi_search.csv` - ROI search results
- `output/heatmaps/heatmap_*.png` - ROI similarity heatmaps
- `output/heatmaps/heatmap_*_overlay.png` - Heatmaps overlaid on thumbnails
- `output/ot_heatmaps/ot_heatmap_*.png` - OT per-patch similarity heatmaps
- `output/ot_heatmaps/ot_overlay_*.png` - OT heatmaps overlaid on thumbnails
- `output/bayesian/disease_probs_{slide_id}.json` - Bayesian disease probabilities
- `output/bayesian/disease_probs_{slide_id}.csv` - Probability matrix CSV
- `output/llm/slide_{slide_id}.json` - LLM-ready JSON payload with neighbor metadata
- `output/llm/summary_{slide_id}.txt` - LLM-generated clinical summary

### Key Data Files
- `data/slide_dimensions.csv` - Slide dimensions for accurate heatmap coordinate mapping
- `data/mosaic_metadata.pkl` - Mosaic cluster locations for full tissue coverage
- `data/thumbnails/` - Slide thumbnails for overlay visualization
- `data/final_scan_id_disease_map_rich_data.csv` - Rich metadata for LLM summarization

### Next Steps
- Process more slides by changing `SLIDE_PATH`
- Try different embedding methods: `bob`, `ot`
- Adjust Bayesian parameters (`n_neighbors`, `temperature`) for different probability sharpness
- Customize the LLM system prompt in `prosper.llm.summarizer` for your use case

In [ ]:
print("Demo complete!")
print(f"\nGenerated files:")
print(f"  - Slide search: {slide_search_path}")
print(f"  - ROI search: {roi_search_path}")
if 'result_paths' in dir() and result_paths:
    print(f"  - ROI Heatmap: {result_paths.get('heatmap', 'N/A')}")
    print(f"  - ROI Heatmap overlay: {result_paths.get('overlay', 'N/A')}")
if 'ot_overlays' in dir() and ot_overlays:
    print(f"  - OT heatmaps: {OT_HEATMAP_DIR}")
    for label, data in ot_overlays.items():
        print(f"    - {label} slide {data['sid']}: overlay generated")
if 'bayesian_json_path' in dir():
    print(f"  - Bayesian probabilities: {bayesian_json_path}")
if 'llm_json_path' in dir():
    print(f"  - LLM payload: {llm_json_path}")
if 'summary_path' in dir() and summary_path.exists():
    print(f"  - LLM summary: {summary_path}")